# Optimizers Experiments

## 1. Клонирование репозитория

In [ ]:
!git clone https://github.com/irinszn/DL_Optimizers_Experiments.git
%cd DL_Optimizers_Experiments

## 2. Установка зависимостей

In [ ]:
!pip install -q uv
!uv sync

In [ ]:
!pip install -q torch_optimizer seedbank mlflow optuna pyngrok scipy

## 3. Подключение Google Drive и MLflow

In [ ]:
from google.colab import drive, userdata
from pyngrok import ngrok

drive.mount('/content/drive')

import mlflow
mlflow.set_tracking_uri('file:/content/drive/MyDrive/mlflow')

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

import subprocess
subprocess.Popen(['mlflow', 'ui', '--host', '0.0.0.0',
                  '--backend-store-uri', 'file:/content/drive/MyDrive/mlflow',
                  '--port', '5001'])

public_url = ngrok.connect(5001)
print('MLflow UI доступен по адресу:', public_url)

## 4. Загрузка датасета с Kaggle

In [ ]:
import json
import os

os.makedirs('/root/.kaggle', exist_ok=True)

api_token = {"username": "", "key": ""}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(api_token, f)

!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d alessiocorrado99/animals10
!unzip -q animals10.zip

## 5. Генерация зашумлённых датасетов

In [ ]:
from src.data.noises import GaussianNoiseAdder, SaltAndPepperNoiseAdder
from src.data.processing import generate_datasets_on_drive

NOISE_REGISTRY = {
    "GaussianNoiseAdder": GaussianNoiseAdder,
    "SaltAndPepperNoiseAdder": SaltAndPepperNoiseAdder,
}

generate_datasets_on_drive(config_path='configs/base_config.yaml', noise_registry=NOISE_REGISTRY)

## 6. Подбор гиперпараметров (Optuna)

In [ ]:
import yaml
from src.data.processing import get_dataloaders_from_drive
from src.models.simple_cnn import SimpleCNN
from src.training.tuner import HyperparameterTuner

with open('configs/base_config.yaml') as f:
    config = yaml.safe_load(f)

train_loader, val_loader, _ = get_dataloaders_from_drive(
    preprocessed_root_path=config['data']['preprocessed_root_path'],
    scenario_folder_template=config['data']['scenario_folder_template'],
    scenario_name='no_noise',
    random_state=42,
    batch_size=config['training']['batch_size'],
    subset_size=config['data'].get('debug_subset_size'),
)

In [ ]:
sgd_tuner = HyperparameterTuner(
    model_class=SimpleCNN,
    model_params=config['model'].get('params', {}),
    optimizer_name='SGD',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs_per_trial=10,
)
sgd_study = sgd_tuner.tune(n_trials=40)

## 7. Основной эксперимент

In [ ]:
from run import run_experiments

run_experiments()

## 8. Оценка робастности

In [ ]:
from run import run_robustness

run_robustness()